# small\_fable — full pipeline on Colab (GPU) → Hugging Face, with **resume**

Runs SFT → rollouts → GRPO on a free Colab T4 and **pushes checkpoints to Hugging Face**. Both
trainers **checkpoint mid-epoch every ~10 min** (model + optimizer + position) and push to HF, so if
Colab recycles (it kills the runtime after ~4h) you just **re-run the cell** and it resumes from the
last checkpoint — important for the long RL stage, which often dies midway.

**How resume works:** each stage first *pulls* its checkpoint dir from HF (if any), then runs with
`--resume`. A finished stage resumes to a no-op, so re-running top-to-bottom is always safe.

> **Before running:** `Runtime → Change runtime type → T4 GPU`, and add a Hugging Face **write** token
> as a Colab secret named `HF_TOKEN` (🔑 sidebar → *Notebook access* ON;
> https://huggingface.co/settings/tokens ). Then `Runtime → Run all`. If Colab dies, just `Run all`
> again — completed stages skip, the dead stage resumes.


## 0 · GPU check

In [ ]:
!nvidia-smi || echo 'NO GPU — set Runtime → Change runtime type → T4 GPU, then re-run.'


## 1 · Clone the repo & install deps

In [ ]:
import os
REPO = 'https://github.com/sinha-k-prat/small_fable.git'
if not os.path.isdir('small_fable'):
    !git clone -q $REPO
else:
    !cd small_fable && git pull -q
%cd small_fable
!pip install -q -r requirements.txt huggingface_hub
print('\nReady.')


## 2 · Hugging Face setup (token, repo, push/pull helpers)
Logs in with your `HF_TOKEN` secret, creates a **public** repo `<username>/small_fable-planner`, and
exposes `pull_ckpt()` (download a checkpoint to resume) and `push_ckpts()` (used for the rollouts file).
The token is also exported to the environment so the training subprocesses can push their own checkpoints.

In [ ]:
import os
from huggingface_hub import HfApi, create_repo, snapshot_download

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    from huggingface_hub import notebook_login; notebook_login()
    HF_TOKEN = HfApi().token
os.environ['HF_TOKEN'] = HF_TOKEN or ''   # so `!python` trainers can push their checkpoints

api = HfApi(token=HF_TOKEN)
HF_USER = api.whoami()['name']
HF_REPO = f'{HF_USER}/small_fable-planner'
create_repo(HF_REPO, repo_type='model', exist_ok=True, private=False, token=HF_TOKEN)
print('HF repo:', f'https://huggingface.co/{HF_REPO}')
print('>>> Use this HF_REPO in inference_colab.ipynb:', HF_REPO)

def pull_ckpt(sub):
    """Download checkpoint dir <sub> from HF into ./<sub> so a stage can --resume after a restart."""
    try:
        snapshot_download(repo_id=HF_REPO, allow_patterns=[f'{sub}/*'], local_dir='.')
        ok = os.path.exists(os.path.join(sub, 'train_state.pt'))
        print(f'[pull] {sub}: ' + ('resumable checkpoint found' if ok else 'no resumable state — fresh start'))
    except Exception as e:
        print(f'[pull] {sub}: nothing to resume ({e})')

def push_ckpts(reason='update'):
    for fn in ['rl_rollouts.jsonl']:
        if os.path.exists(fn):
            api.upload_file(path_or_fileobj=fn, path_in_repo=fn, repo_id=HF_REPO,
                            commit_message=f'{reason}: {fn}')
            print(f'[hf] pushed {fn}')


## 3 · Stage 1 — Joint SFT  *(checkpoints + resumes via HF)*
Held-out validation every epoch (`held {...}`): `plan_ce`/`resp_ce` fall, `ablation_gap` stays positive.
`--ckpt_every_min 10` saves model+optimizer to HF every ~10 min; `--resume` continues from the last one.

In [ ]:
pull_ckpt('joint_ckpt')   # resume if a partial SFT checkpoint exists on HF
!python -u train_sft.py \
    --data dataset/sft_100.jsonl --train 70 --held 30 \
    --epochs 6 --lr 2e-5 --bs 4 --max_resp 64 --probe 12 \
    --out joint_ckpt --device cuda \
    --resume --hf_repo $HF_REPO --ckpt_every_min 10


## 4 · Stage 2a — Offline rollouts
One pass (no mid-checkpoint); ~10–20 min. If it dies, just re-run — it regenerates `rl_rollouts.jsonl`.

In [ ]:
pull_ckpt('joint_ckpt')   # need the SFT checkpoint locally to sample from
!python -u rollout_offline.py \
    --sft_ckpt joint_ckpt --data dataset/sft_100.jsonl \
    --train 80 --group 8 --temp 1.5 --top_p 0.98 --max_resp 64 \
    --out rl_rollouts.jsonl --device cuda
push_ckpts('after-rollouts')   # cache the rollouts on HF


## 5 · Stage 2b — Off-policy GRPO  *(checkpoints + resumes via HF)*
**This is the stage most likely to be interrupted.** It checkpoints mid-inner-epoch every ~10 min
(model + optimizer + `(inner_epoch, group_idx)`) to HF. On a restart, re-run this cell: it pulls the
latest `rl_ckpt`, restores the optimizer, and continues from the exact group it left off.
Watch `held_reward=` each inner epoch; final `|ΔL2| > 0` proves RL moved the adapter.

In [ ]:
pull_ckpt('joint_ckpt')   # SFT init (used only if no rl_ckpt to resume)
pull_ckpt('rl_ckpt')      # resume the RL run if a partial checkpoint exists
# rollouts must exist locally; regenerate (Stage 2a) if this is a fresh runtime
import os; assert os.path.exists('rl_rollouts.jsonl'), 'run Stage 2a first (rollouts not on disk)'
!python -u train_grpo_offline.py \
    --rollouts rl_rollouts.jsonl --sft_ckpt joint_ckpt --data dataset/sft_100.jsonl \
    --out rl_ckpt --inner_epochs 3 --lr 1e-4 --clip_eps 0.2 \
    --beta_plan 1.0 --beta_ce 0.1 --held 16 --device cuda \
    --resume --hf_repo $HF_REPO --ckpt_every_min 10
print('All checkpoints on HF:', f'https://huggingface.co/{HF_REPO}/tree/main')


## 6 · Compare — base vs SFT vs SFT+RL
`--sample` (temp 0.7) is required to see small RL effects.

In [ ]:
!python -u compare.py \
    --sft_ckpt joint_ckpt --rl_ckpt rl_ckpt \
    --data dataset/sft_100.jsonl --train 80 --held 20 \
    --sample --device cuda


## 7 · Head-to-head on a brand-new hard prompt
A multi-step prompt that appears **nowhere** in the data. (For standalone inference loading straight
from Hugging Face, use `inference_colab.ipynb`.)

In [ ]:
import torch
from model_joint import JointModel, decode_plan
HARD_PROMPT = ('A snail is at the bottom of a 12-meter well. Each day it climbs 4 meters, '
               'but each night it slides back 3 meters. On which day does it first reach the top?')
def run(ckpt, label, n=3, temp=0.7):
    m = JointModel.from_checkpoint('Qwen/Qwen2.5-1.5B-Instruct', ckpt, device='cuda'); m.eval()
    print(f'\n{"="*68}\n{label}  ({ckpt})\n{"="*68}')
    with torch.no_grad():
        for k in range(n):
            p_ids, p_attn = m.batch_prompts([HARD_PROMPT])
            plan = m.sample_plan(p_ids, p_attn, temp=temp, sample=True)
            gen = m.generate_answer(p_ids, p_attn, plan, temp=temp, sample=True, max_new_tokens=64)
            print(f'  sample {k+1}  plan: {" → ".join(decode_plan(plan[0]))}')
            print(f'            answer: {m.tok.decode(gen[0], skip_special_tokens=True).strip()}')
    del m; torch.cuda.empty_cache()
print('HARD PROMPT (not in data):\n ', HARD_PROMPT)
run('joint_ckpt', 'SFT only'); run('rl_ckpt', 'SFT + GRPO')
